In [ ]:
import class_ML_carbon as carb
import numpy as np
from netCDF4 import Dataset
from tensorflow import keras
from matplotlib import pyplot as plt

# read in your data in a netCDF format. The data are in the form of datapoints x features. they are stored in two arrays: "inputs" and "outputs". This file has been saved for separate years and compressed due to GitHub requirements, it would need to be rebuild. 

In [ ]:
inpfile = Dataset("./data/Free_run_2016-2020_data.nc")   

In [ ]:
inputs = inpfile.variables["inputs"][:].transpose()   
outputs = inpfile.variables["outputs"][:].transpose()

# select the test subset from the data, we assume the test data are the last 20% of the data in the sequence, the first 80% were used for training and validation

In [ ]:
inputs_train = inputs[:round(0.8*inputs.shape[0]),:]
outputs_train = outputs[:round(0.8*inputs.shape[0]),:]
inputs_test = inputs[round(0.8*inputs.shape[0]):,:]
outputs_test = outputs[round(0.8*inputs.shape[0]):,:]

# read the existing model architecture and weights

In [ ]:
model = keras.models.load_model('./model')

# use the model to predict the test data 

In [ ]:
pred_init = carb.ML_carbon(inputs = inputs_test, normalization_inputs=inputs_train, model = model)
pred = pred_init.predicted_values()

# compare the test data with the predicted data and plot them variable after variable (alternatively you can plot them in multi-panel plot.

In [ ]:
pred = carb.invert_normalization(data_norm=pred, data_ref=outputs_train)
outputs_list = pred_init.provide_output_names()

for i, output in enumerate(outputs_list):
    plt.figure(figsize=(8, 6))
    hb = plt.hexbin(pred[:,i], outputs_test[:,i], gridsize=50, cmap='plasma')
    plt.colorbar(hb, label='Point density')
    lims = [
        np.min([plt.xlim(), plt.ylim()]),  # min of both axes
        np.max([plt.xlim(), plt.ylim()]),  # max of both axes
    ]
    plt.plot(lims, lims, 'k--', alpha=0.75, zorder=0)  # black dashed line
    plt.xlim(lims)
    plt.ylim(lims)
    plt.xlabel('predicted')
    plt.ylabel('test data')
    plt.title(output)
    plt.show()